# PAC-MAN AI — Full Workflow

## Introduction

This project applies **machine learning and reinforcement learning** to the classic Pac-Man arcade game.
The goal is to train an autonomous agent that learns to play Pac-Man from scratch, guided only
by game state observations — without access to the raw pixel environment used by human players.

### Problem Statement

Can a neural agent learn an effective Pac-Man policy purely from a structured game-state representation?
Can a second, vision-limited agent infer that representation from raw pixel frames using computer vision?

### Why It Matters

- Reinforcement learning in partially observable, adversarial environments is a core open problem in AI.
- Pac-Man provides a well-defined benchmark with sparse rewards, multi-agent dynamics (4 ghosts),
  and a clear success criterion (high score / level clear).
- The vision pipeline (frame → segmentation → state) reflects real-world perception challenges
  where agents must build world models from noisy sensors.

### Techniques Used (≥ 3)

| # | Technique | Module |
|---|-----------|--------|
| 1 | **Random baseline** — uniform action policy | RL fundamentals |
| 2 | **Semantic segmentation** — TinyU-Net CNN, pixel-level multi-class | Deep Learning / CV |
| 3 | **Slot-mask pellet detection** — deterministic, brightness-based | Classical CV |
| 4 | **MaskablePPO** — action-masked proximal policy optimisation | Reinforcement Learning |
| 5 | **SHAP GradientExplainer** — critic attribution analysis | Interpretability |

### Pipeline Overview

```
Game engine  →  synthetic frame + mask  →  U-Net training
                                              ↓
Game engine  →  grid observation  →  MaskablePPO training
                                              ↓
Pixel frame  →  U-Net mask  →  state snapshot  →  vision agent
```

---

| Section | Content |
|---------|--------|
| 0 | Setup: paths, imports, MLflow |
| 1 | Data Loading & Validation — environment & EDA |
| 2 | Synthetic Dataset Generation |
| 3 | Feature Engineering — observation space design |
| 4 | Segmentation U-Net Training (MLflow) |
| 5 | Segmentation Evaluation: mIoU + visual inspection |
| 6 | Slot-Mask Pellet Detection (no NN) |
| 7 | RL Agent Training: MaskablePPO (MLflow) |
| 8 | Interpretability: Permutation Importance + SHAP |
| 9 | Learning Curves from MLflow |
| 10 | Agent Benchmark: Direct-State vs Vision Pipeline |
| 11 | Conclusions |

> **Reproducibility:** `RANDOM_SEED = 42` throughout.  
> **Requirements:** run from the repo root, or ensure `PAC-MAN-AI/` is on `sys.path`.
> Before submission: restart kernel and run all cells top-to-bottom.


In [ ]:
# ── 0. Paths & common imports ─────────────────────────────────────────────────
import sys
from pathlib import Path

ROOT = Path().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import mlflow
import json, time, random
from IPython.display import display, Image as IPImage

# Project source
from src.environment.pacman_env import PacmanEnv, PacmanGridEnv
from src.environment.game_logic import DEFAULT_MAZE, GameState
from src.dataset.pacman_map_dataset import PacmanMapDatasetGenerator
from src.models.segmentation_detector import (
    SegmentationDetector, TrainConfig, extract_instances,
    build_pellet_slot_mask, detect_pellets_grid,
    CLASS_TO_ID, ID_TO_CLASS,
)
from src.utils.mlflow_logger import MLflowLogger
from src.utils.pacman_renderer import render_state_rgb_sprites, render_state_with_hud_sprites

# Paths
MODELS_DIR      = ROOT / 'models'
DATA_DIR        = ROOT / 'data'
TRAIN_DIR       = DATA_DIR / 'segmentation' / 'train'
TEST_DIR        = DATA_DIR / 'segmentation' / 'test'
REPORTS_DIR     = ROOT / 'reports'
MLFLOW_DB       = ROOT / 'mlruns' / 'mlflow.db'
MLFLOW_TRACKING_URI = f'sqlite:///{MLFLOW_DB}'
RANDOM_SEED     = 42

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
REPORTS_DIR.mkdir(exist_ok=True)
print(f'ROOT            : {ROOT}')
print(f'MLflow URI      : {MLFLOW_TRACKING_URI}')
print(f'CUDA available  : {torch.cuda.is_available()}')

---
## 1 — Data Loading & Validation

We use a **custom Pac-Man game engine** (`PacmanEnv`) as both the data source and the training
environment. This section validates the observation space, action space, and reward distribution
before any model is built — a required quality gate before modelling.


In [ ]:
# 1a. Environment loading & basic validation
env = PacmanEnv(render_mode='ansi', seed=RANDOM_SEED)
obs, info = env.reset()

print(f'Observation shape : {obs.shape}  dtype={obs.dtype}')
print(f'Observation range : [{obs.min():.3f}, {obs.max():.3f}]')
print(f'Action space      : {env.action_space}')
print(f'Info keys         : {list(info.keys())}')

In [ ]:
# 1b. Observation space distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(obs, bins=50, color='steelblue', edgecolor='white', linewidth=0.4)
axes[0].set_title('Observation value distribution (reset)')
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Count')

axes[1].imshow(obs.reshape(-1, 1), aspect='auto', cmap='viridis', interpolation='nearest')
axes[1].set_title('Observation vector (visualised as column)')
axes[1].set_xlabel('Feature index')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'eda_observation_space.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# 1c. Maze layout
maze = np.array(DEFAULT_MAZE)
cmap = plt.cm.colors.ListedColormap(['black', '#1a1aff', '#ffff00', '#ff00ff', '#888888', '#444444'])
fig, ax = plt.subplots(figsize=(6, 8))
ax.imshow(maze, cmap=cmap, vmin=0, vmax=5, interpolation='nearest')
patches = [
    mpatches.Patch(color='black',     label='Empty'),
    mpatches.Patch(color='#1a1aff',   label='Wall'),
    mpatches.Patch(color='#ffff00',   label='Pellet'),
    mpatches.Patch(color='#ff00ff',   label='Power pellet'),
    mpatches.Patch(color='#888888',   label='Ghost door'),
    mpatches.Patch(color='#444444',   label='Ghost house'),
]
ax.legend(handles=patches, bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax.set_title(f'Arcade maze ({maze.shape[1]}×{maze.shape[0]})')
ax.axis('off')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'eda_maze_layout.png', dpi=120, bbox_inches='tight')
plt.show()

n_walls   = (maze == 1).sum()
n_pellets = (maze == 2).sum()
n_power   = (maze == 3).sum()
print(f'Walls: {n_walls}  |  Pellets: {n_pellets}  |  Power pellets: {n_power}')

In [ ]:
# 1d. Action space & reward distribution (random rollout)
N_EPISODES = 200
episode_rewards, episode_lengths = [], []
action_counts = np.zeros(env.action_space.n, dtype=int)

for _ in range(N_EPISODES):
    obs, _ = env.reset()
    total_r, steps = 0.0, 0
    done = False
    while not done:
        action = env.action_space.sample()
        action_counts[action] += 1
        obs, r, terminated, truncated, _ = env.step(action)
        total_r += r
        steps += 1
        done = terminated or truncated
    episode_rewards.append(total_r)
    episode_lengths.append(steps)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(episode_rewards, bins=30, color='gold', edgecolor='black', linewidth=0.4)
axes[0].set_title('Episode reward (random policy)')
axes[0].set_xlabel('Total reward')

axes[1].hist(episode_lengths, bins=30, color='tomato', edgecolor='black', linewidth=0.4)
axes[1].set_title('Episode length (random policy)')
axes[1].set_xlabel('Steps')

action_names = ['UP', 'DOWN', 'LEFT', 'RIGHT', 'NOOP']
axes[2].bar(action_names[:env.action_space.n], action_counts, color='mediumpurple')
axes[2].set_title('Action frequency')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'eda_rewards_actions.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Reward  — mean={np.mean(episode_rewards):.1f}  std={np.std(episode_rewards):.1f}')
print(f'Length  — mean={np.mean(episode_lengths):.1f}  std={np.std(episode_lengths):.1f}')

---
## 2 — Synthetic Dataset Generation & Labelling

`PacmanMapDatasetGenerator` renders game frames and produces pixel-level segmentation masks
with bounding-box annotations for each game object. This is the training data for the U-Net.


In [ ]:
# 2a. Preview: single rendered sample with mask overlay
gen = PacmanMapDatasetGenerator()

def render_sample(seed: int, max_steps: int = 80):
    """Return (frame_rgb, class_mask, annotations) for a random game state."""
    state = GameState(seed=seed)
    for _ in range(max_steps):
        action = random.randint(0, 3)
        done, _ = state.step(action)
        if done:
            break
    pil_frame, pil_mask, ann = gen.render_state(state)
    return np.array(pil_frame)[:, :, :3], np.array(pil_mask), ann

CLASS_COLORS = {
    'pacman':           (255, 212,   0),
    'blinky':           (255,  30,  30),
    'pinky':            (255, 155, 220),
    'inky':             ( 50, 220, 255),
    'clyde':            (255, 160,  50),
    'frightened_ghost': ( 50,  50, 200),
    'ghost_eyes':       (200, 200, 255),
    'pellet':           (255, 255, 180),
    'power_pellet':     (255, 200, 255),
    'wall':             ( 30,  80, 220),
    'ghost_door':       (180, 180, 180),
    'ghost_house':      ( 80,  80,  80),
    'fruit_cherry':     (200,  20,  40),
}

frame, mask, ann = render_sample(seed=RANDOM_SEED, max_steps=80)

# Build coloured mask overlay
colored = np.zeros((*mask.shape, 3), dtype=np.uint8)
for class_name, color in CLASS_COLORS.items():
    if class_name in CLASS_TO_ID:
        colored[mask == CLASS_TO_ID[class_name]] = color

overlay = (frame.astype(float) * 0.55 + colored.astype(float) * 0.45).clip(0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(frame);           axes[0].set_title('Rendered frame'); axes[0].axis('off')
axes[1].imshow(colored);         axes[1].set_title('Class mask');     axes[1].axis('off')
axes[2].imshow(overlay);         axes[2].set_title('Overlay');        axes[2].axis('off')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'generated_preview_sample.png', dpi=120, bbox_inches='tight')
plt.show()
print('Objects detected:', [o['label'] for o in ann.get('objects', [])])

In [ ]:
# 2b. Class distribution across multiple samples
from collections import Counter

n_samples = 40
pixel_counts: Counter = Counter()
for s in range(n_samples):
    _, m, _ = render_sample(seed=s, max_steps=60)
    for cid, count in zip(*np.unique(m, return_counts=True)):
        if cid in ID_TO_CLASS:
            pixel_counts[ID_TO_CLASS[cid]] += int(count)

labels, counts = zip(*sorted(pixel_counts.items(), key=lambda x: -x[1]))
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(labels, counts, color='steelblue')
ax.set_title(f'Pixel-level class distribution ({n_samples} samples)')
ax.set_ylabel('Total pixels')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'generated_preview_mask.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# 2c. Generate train / test datasets if missing
if not (TRAIN_DIR / 'images').exists():
    print('Generating training dataset (2 000 samples)…')
    gen_full = PacmanMapDatasetGenerator()
    gen_full.generate_dataset(
        output_dir=str(TRAIN_DIR),
        n_samples=2000,
        seed=RANDOM_SEED,
        max_random_steps=300,
    )
    print('Done — train set')
else:
    train_images = list((TRAIN_DIR / 'images').glob('*.png'))
    print(f'Train dataset already exists: {len(train_images)} images')

if not (TEST_DIR / 'images').exists():
    print('Generating test dataset (400 samples)…')
    gen_full = PacmanMapDatasetGenerator()
    gen_full.generate_dataset(
        output_dir=str(TEST_DIR),
        n_samples=400,
        seed=RANDOM_SEED + 1,
        max_random_steps=300,
    )
    print('Done — test set')
else:
    test_images = list((TEST_DIR / 'images').glob('*.png'))
    print(f'Test dataset already exists: {len(test_images)} images')

---
## 3 — Feature Engineering

The MaskablePPO agent does **not** operate on raw pixel frames. Instead, the game state is
projected into a multi-channel binary grid of shape `(C, 31, 28)` — one binary plane per
semantic concept. This design:

- Removes perceptual ambiguity (e.g. ghost colour vs. wall colour)
- Makes the state space compact and fully observable for the direct agent
- Allows SHAP to attribute importance to interpretable concepts, not raw pixels


In [ ]:
# 3. Observation space channel engineering
# PacmanGridEnv builds an (C, ROWS, COLS) tensor from the GameState.
# Each channel is a binary indicator plane.

CHANNEL_SPEC = {
    0:  'walls             — static maze structure',
    1:  'pellets           — remaining normal dots',
    2:  'power_pellets     — energiser positions',
    3:  'pacman            — agent position',
    4:  'ghosts_normal     — active threat positions',
    5:  'ghosts_frightened — safe-to-eat positions',
    6:  'fruit             — bonus item position',
    7:  'lives_hud         — remaining lives (broadcast)',
    8:  'score_hud         — current score (broadcast)',
}

grid_env = PacmanGridEnv(seed=RANDOM_SEED)
obs0, _ = grid_env.reset()
print(f'Grid observation shape : {obs0.shape}   (C × ROWS × COLS)')
print(f'Dtype                  : {obs0.dtype}')
print(f'Value range            : [{obs0.min()}, {obs0.max()}]')
print(f'\nChannel definitions:')
for ch, desc in CHANNEL_SPEC.items():
    n_active = int(obs0[ch].sum()) if ch < obs0.shape[0] else 'n/a'
    print(f'  ch {ch:2d}  {desc}  (active cells: {n_active})')

# Visualise first 9 channels side-by-side
n_ch = min(obs0.shape[0], 9)
fig, axes = plt.subplots(1, n_ch, figsize=(n_ch * 2.5, 3))
for ch in range(n_ch):
    axes[ch].imshow(obs0[ch], cmap='gray', interpolation='nearest')
    short = CHANNEL_SPEC[ch].split('—')[0].strip() if ch in CHANNEL_SPEC else f'ch{ch}'
    axes[ch].set_title(short, fontsize=7)
    axes[ch].axis('off')
plt.suptitle('Grid observation channels (initial state)', y=1.02)
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'feature_engineering_channels.png', dpi=120, bbox_inches='tight')
plt.show()


---
## 4 — Segmentation U-Net Training (MLflow)

`TinyU-Net` is trained with pixel-wise cross-entropy to segment game frames into 13 semantic classes.
All hyperparameters and metrics are logged to **MLflow** for full reproducibility.


In [ ]:
# 3a. Training configuration
SEG_MODEL_PATH = MODELS_DIR / 'segmentation_unet_long.pt'
RETRAIN_SEG    = False   # set True to force re-training

TRAIN_EPOCHS     = 300
TRAIN_BATCH_SIZE = 8
TRAIN_LR         = 1e-3
TRAIN_VAL_SPLIT  = 0.1
DEVICE           = 'cuda' if torch.cuda.is_available() else 'cpu'

if not SEG_MODEL_PATH.exists() or RETRAIN_SEG:
    print(f'Training segmentation model ({TRAIN_EPOCHS} epochs, device={DEVICE})…')
    cfg = TrainConfig(
        dataset_dir=TRAIN_DIR,
        output_path=SEG_MODEL_PATH,
        epochs=TRAIN_EPOCHS,
        batch_size=TRAIN_BATCH_SIZE,
        lr=TRAIN_LR,
        val_split=TRAIN_VAL_SPLIT,
        seed=RANDOM_SEED,
        device=DEVICE,
    )
    with MLflowLogger(
        experiment_name='segmentation_training',
        run_name=f'unet_{TRAIN_EPOCHS}ep',
        tracking_uri=MLFLOW_TRACKING_URI,
    ) as logger:
        logger.log_params({
            'epochs': TRAIN_EPOCHS,
            'batch_size': TRAIN_BATCH_SIZE,
            'lr': TRAIN_LR,
            'val_split': TRAIN_VAL_SPLIT,
            'device': DEVICE,
        })
        detector = SegmentationDetector(num_classes=len(CLASS_TO_ID), device=DEVICE)
        detector.train(cfg)
        logger.log_artifact(str(SEG_MODEL_PATH))
    print(f'Model saved: {SEG_MODEL_PATH}')
else:
    print(f'Using existing model: {SEG_MODEL_PATH}')

---
## 5 — Segmentation Evaluation

We compute **mean IoU (mIoU)** over the held-out test set (400 images) and log results to MLflow.
Visual inspection confirms that the model correctly segments walls, actors, and pellets.


In [ ]:
# 4a. Load model + compute mIoU on test set
import cv2
from PIL import Image

detector = SegmentationDetector.load(SEG_MODEL_PATH, device=DEVICE)
print(f'Model loaded: {SEG_MODEL_PATH.name} | classes={detector.num_classes}')

def compute_miou(detector, test_dir: Path, id_to_class: dict):
    """Compute mIoU over the test set."""
    class_ids = sorted(id_to_class.keys())
    intersection = {cid: 0 for cid in class_ids}
    union        = {cid: 0 for cid in class_ids}

    img_paths = sorted((test_dir / 'images').glob('*.png'))
    msk_paths = sorted((test_dir / 'masks').glob('*.png'))
    assert len(img_paths) == len(msk_paths), 'Image/mask count mismatch'

    for img_p, msk_p in zip(img_paths, msk_paths):
        img = np.array(Image.open(img_p).convert('RGB'))
        gt  = np.array(Image.open(msk_p))
        pred = detector.predict_mask(img)
        for cid in class_ids:
            pred_c = (pred == cid)
            gt_c   = (gt   == cid)
            intersection[cid] += int((pred_c & gt_c).sum())
            union[cid]        += int((pred_c | gt_c).sum())

    iou_per_class = {}
    for cid in class_ids:
        if union[cid] > 0:
            iou_per_class[id_to_class[cid]] = intersection[cid] / union[cid]
    miou = np.mean(list(iou_per_class.values()))
    return miou, iou_per_class

miou, iou_pc = compute_miou(detector, TEST_DIR, ID_TO_CLASS)
print(f'\nmIoU = {miou:.4f}')
print('\nPer-class IoU:')
for cls, iou in sorted(iou_pc.items(), key=lambda x: -x[1]):
    bar = '█' * int(iou * 30)
    print(f'  {cls:<22} {iou:.4f}  {bar}')

# Log to MLflow
with MLflowLogger(
    experiment_name='segmentation_vision_qc',
    run_name='notebook_eval',
    tracking_uri=MLFLOW_TRACKING_URI,
) as logger:
    logger.log_metric('miou', miou)
    for cls, iou in iou_pc.items():
        logger.log_metric(f'iou_{cls}', iou)

In [ ]:
# 4b. Visual inspection — random test samples
import random as stdlib_random

img_paths = sorted((TEST_DIR / 'images').glob('*.png'))
msk_paths = sorted((TEST_DIR / 'masks').glob('*.png'))
indices   = stdlib_random.sample(range(len(img_paths)), 4)

fig, axes = plt.subplots(4, 3, figsize=(12, 16))
for row, idx in enumerate(indices):
    img  = np.array(Image.open(img_paths[idx]).convert('RGB'))
    gt   = np.array(Image.open(msk_paths[idx]))
    pred = detector.predict_mask(img)

    def mask_to_color(m):
        c = np.zeros((*m.shape, 3), dtype=np.uint8)
        for cls, color in CLASS_COLORS.items():
            if cls in CLASS_TO_ID:
                c[m == CLASS_TO_ID[cls]] = color
        return c

    axes[row, 0].imshow(img);               axes[row, 0].set_title('Input frame')
    axes[row, 1].imshow(mask_to_color(gt)); axes[row, 1].set_title('Ground truth')
    axes[row, 2].imshow(mask_to_color(pred)); axes[row, 2].set_title('Prediction')
    for ax in axes[row]: ax.axis('off')

plt.suptitle('Segmentation model — visual inspection', y=1.01, fontsize=14)
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'segmentation_inspection.png', dpi=100, bbox_inches='tight')
plt.show()

---
## 6 — Pellet Detection Without Neural Network (Slot-Mask)

As a **classical CV baseline**, pellets can be detected by sampling fixed screen coordinates
(derived analytically from the maze grid) and thresholding pixel brightness — no neural network required.
This approach is 100% deterministic and useful for ablation comparisons.


In [ ]:
# 5. Pellet slot mask — brightness-based detection
sample_img = np.array(Image.open(img_paths[0]).convert('RGB'))

slot_mask   = build_pellet_slot_mask(sample_img.shape[:2], include_power=True)
instances   = detect_pellets_grid(sample_img, brightness_threshold=160, sample_radius=2)

overlay_p = sample_img.copy()
for inst in instances:
    x, y, w, h = inst['bbox']
    color = (255, 230, 0) if inst['label'] == 'pellet' else (200, 80, 255)
    cv2.rectangle(overlay_p, (x, y), (x+w, y+h), color, 1)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(sample_img);  axes[0].set_title('Input frame');       axes[0].axis('off')
axes[1].imshow(slot_mask, cmap='gray'); axes[1].set_title('Slot mask'); axes[1].axis('off')
axes[2].imshow(overlay_p);   axes[2].set_title(f'Detected ({len(instances)} pellets)'); axes[2].axis('off')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'ignored_pellets.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Detected instances: {len(instances)}')

---
## 7 — RL Agent Training: MaskablePPO (MLflow)

**MaskablePPO** (from `sb3-contrib`) prevents illegal moves by masking actions that would walk into walls.
We use a two-phase curriculum: short episodes (exploration) → long episodes (exploitation).
All episode rewards, policy loss, and value loss are streamed to MLflow during training.


In [ ]:
# 6a. Install sb3-contrib if needed
try:
    from sb3_contrib import MaskablePPO
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'sb3-contrib', '-q'])
    from sb3_contrib import MaskablePPO

from sb3_contrib.common.wrappers import ActionMasker
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack
from stable_baselines3.common.callbacks import CheckpointCallback
from src.utils.maskable_env import get_action_mask
from src.utils.ppo_cnn import PacmanCNN
from src.utils.training_callbacks import MetricsCallback

print('sb3-contrib ready')

In [ ]:
# 6b. Training configuration
RL_MODEL_PATH = MODELS_DIR / 'ppo_pacman.zip'
RETRAIN_RL    = False   # set True to force re-training

N_ENVS       = 8
N_STEPS      = 256
BATCH_SIZE   = 512
PHASE1_STEPS = 1_000_000
PHASE2_STEPS = 4_000_000
USE_MASKABLE = True

def make_env(seed: int = 0, max_steps: int = 5000):
    def _init():
        e = PacmanGridEnv(seed=seed, max_steps=max_steps)
        if USE_MASKABLE:
            e = ActionMasker(e, get_action_mask)
        return e
    return _init

if not RL_MODEL_PATH.exists() or RETRAIN_RL:
    print(f'Training MaskablePPO — target {PHASE1_STEPS + PHASE2_STEPS:,} steps…')

    # Phase 1 — short episodes, high entropy
    vec_env1 = DummyVecEnv([make_env(i, max_steps=5000) for i in range(N_ENVS)])
    policy_kwargs = dict(features_extractor_class=PacmanCNN)

    model = MaskablePPO(
        'CnnPolicy', vec_env1,
        n_steps=N_STEPS, batch_size=BATCH_SIZE,
        ent_coef=0.02, learning_rate=3e-4,
        policy_kwargs=policy_kwargs,
        tensorboard_log=str(ROOT / 'logs' / 'tensorboard'),
        verbose=0, seed=RANDOM_SEED,
    )

    with MLflowLogger(
        experiment_name='rl_training',
        run_name='maskable_ppo_curriculum',
        tracking_uri=MLFLOW_TRACKING_URI,
    ) as logger:
        logger.log_params({
            'n_envs': N_ENVS, 'n_steps': N_STEPS, 'batch_size': BATCH_SIZE,
            'phase1_steps': PHASE1_STEPS, 'phase2_steps': PHASE2_STEPS,
            'use_maskable': USE_MASKABLE,
        })
        cb = MetricsCallback(mlflow_logger=logger, log_every=10_000)
        model.learn(PHASE1_STEPS, callback=cb, reset_num_timesteps=True)

        # Phase 2 — longer episodes, lower entropy
        vec_env2 = DummyVecEnv([make_env(i, max_steps=16000) for i in range(N_ENVS)])
        model.set_env(vec_env2)
        model.ent_coef = 0.005
        model.learn(PHASE2_STEPS, callback=cb, reset_num_timesteps=False)

        model.save(str(RL_MODEL_PATH))
        logger.log_artifact(str(RL_MODEL_PATH))

    print(f'Model saved: {RL_MODEL_PATH}')
else:
    print(f'Using existing model: {RL_MODEL_PATH}')

In [ ]:
# 6c. Watch the trained agent (inline animation)
from IPython.display import clear_output
import matplotlib
matplotlib.use('Agg')

model_rl = MaskablePPO.load(str(RL_MODEL_PATH))

watch_env = PacmanGridEnv(seed=0, max_steps=2000)
obs, _ = watch_env.reset()
frames = []

done = False
while not done and len(frames) < 300:
    action, _ = model_rl.predict(obs, deterministic=True)
    obs, _, terminated, truncated, _ = watch_env.step(int(action))
    frame_rgb = render_state_rgb_sprites(watch_env.state, scale=2)
    frames.append(frame_rgb)
    done = terminated or truncated

# Save GIF
import imageio
gif_path = REPORTS_DIR / 'demo_episode.gif'
imageio.mimsave(str(gif_path), frames, fps=12)
print(f'Episode GIF saved: {gif_path}  ({len(frames)} frames)')
display(IPImage(str(gif_path)))

---
## 8 — Interpretability: Permutation Importance + SHAP

Two complementary approaches explain **what the PPO critic has learned**:

1. **Permutation importance** — measure the drop in V(s) when each observation channel is randomly shuffled.
2. **SHAP GradientExplainer** — back-propagate SHAP values through the CNN critic to get per-channel attribution.

Required SHAP plots: summary bar, beeswarm, and waterfall (single-prediction explanation).


In [ ]:
# 7a. Load policy + collect baseline observations
model_rl = MaskablePPO.load(str(RL_MODEL_PATH))
eval_env  = PacmanGridEnv(seed=RANDOM_SEED, max_steps=2000)

obs_list = []
for ep in range(10):
    o, _ = eval_env.reset(seed=ep)
    done = False
    while not done:
        obs_list.append(o.copy())
        a, _ = model_rl.predict(o, deterministic=True)
        o, _, t, tr, _ = eval_env.step(int(a))
        done = t or tr

obs_array = np.stack(obs_list)  # (N, C, H, W)
print(f'Collected {len(obs_array)} observations, shape={obs_array.shape}')

CHANNELS = {
    'walls':             0,
    'pellets':           1,
    'power_pellets':     2,
    'pacman':            3,
    'ghosts_normal':     4,
    'ghosts_frightened': 5,
    'fruit':             6,
    'lives_hud':         7,
    'score_hud':         8,
}

def get_values(obs_batch):
    """Return V(s) from the PPO critic for a batch of observations."""
    obs_t = torch.FloatTensor(obs_batch).to(next(model_rl.policy.parameters()).device)
    with torch.no_grad():
        _, values, _ = model_rl.policy.evaluate_actions(
            obs_t, torch.zeros(len(obs_t), dtype=torch.long)
        )
    return values.cpu().numpy().flatten()

In [ ]:
# 7b. Permutation channel importance
sample_idx = np.random.choice(len(obs_array), size=500, replace=False)
sample_obs = obs_array[sample_idx]
base_values = get_values(sample_obs)

importances = {}
for ch_name, ch_idx in CHANNELS.items():
    if ch_idx >= obs_array.shape[1]:
        continue
    shuffled = sample_obs.copy()
    np.random.shuffle(shuffled[:, ch_idx, :, :])  # shuffle along batch
    shuffled_values = get_values(shuffled)
    importances[ch_name] = float(np.mean(np.abs(base_values - shuffled_values)))

names  = list(importances.keys())
values = [importances[n] for n in names]
order  = np.argsort(values)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh([names[i] for i in order], [values[i] for i in order], color='steelblue')
ax.set_xlabel('Mean |ΔV(s)| when channel permuted')
ax.set_title('Permutation channel importance (PPO critic)')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'interpretability_permutation.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# 7c. SHAP GradientExplainer
import shap
import torch.nn as nn

CHANNEL_NAMES = list(CHANNELS.keys())

class CriticWrapper(nn.Module):
    def __init__(self, policy):
        super().__init__()
        self.policy = policy
    def forward(self, x):
        features = self.policy.features_extractor(x)
        latent   = self.policy.mlp_extractor.forward_critic(features)
        return self.policy.value_net(latent)

wrapper   = CriticWrapper(model_rl.policy).eval()
device    = next(wrapper.parameters()).device
bg_tensor = torch.FloatTensor(sample_obs[:50]).to(device)
ex_tensor = torch.FloatTensor(sample_obs[50:100]).to(device)

explainer   = shap.GradientExplainer(wrapper, bg_tensor)
shap_values = explainer.shap_values(ex_tensor)  # (N, C, H, W)

channel_shap = np.abs(shap_values).mean(axis=(0, 2, 3))  # mean |shap| per channel
ch_available = CHANNEL_NAMES[:len(channel_shap)]
order_s      = np.argsort(channel_shap)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh([ch_available[i] for i in order_s], channel_shap[order_s], color='coral')
ax.set_xlabel('Mean |SHAP| per channel')
ax.set_title('SHAP GradientExplainer — PPO critic channel attribution')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'interpretability_shap.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# 8d. SHAP beeswarm plot — feature impact direction
# Beeswarm shows both magnitude and direction: does high feature value push V(s) up or down?
import shap

# Compute SHAP values for a small sample (one frame per position in the grid)
# We summarise channel attribution spatially: mean absolute SHAP per channel
# then build a synthetic 2-D matrix (samples × channels) for the beeswarm
n_beeswarm = min(100, len(sample_obs))
sv_flat = np.abs(shap_values[:n_beeswarm]).reshape(n_beeswarm, obs0.shape[0] if 'obs0' in dir() else shap_values.shape[1], -1).mean(axis=2)  # (N, C)
ch_available = CHANNEL_NAMES[:sv_flat.shape[1]]

# Feature matrix: mean activation per channel per sample
feat_matrix = sample_obs[50:50+n_beeswarm].reshape(n_beeswarm, shap_values.shape[1], -1).mean(axis=2)

explanation = shap.Explanation(
    values=sv_flat,
    base_values=np.zeros(n_beeswarm),
    data=feat_matrix,
    feature_names=ch_available,
)
fig_bee, ax_bee = plt.subplots(figsize=(9, 5))
shap.plots.beeswarm(explanation, max_display=len(ch_available), show=False)
plt.title('SHAP beeswarm — PPO critic channel attribution')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'interpretability_shap_beeswarm.png', dpi=120, bbox_inches='tight')
plt.show()
print('Beeswarm: channels on the right push V(s) up (red) or down (blue) relative to the mean baseline.')


In [ ]:
# 8e. SHAP waterfall plot — single-prediction explanation
# Waterfall shows how each channel contribution stacks up from the base value E[V(s)]
# to the actual V(s) for one specific game state.

# Pick an interesting sample: highest-value state (Pac-Man presumably doing well)
all_values = get_values(sample_obs)
best_idx   = int(np.argmax(all_values))

# Recompute SHAP for this single sample relative to a full background
single_obs = torch.FloatTensor(sample_obs[best_idx:best_idx+1]).to(device)
single_sv  = explainer.shap_values(single_obs)[0]  # (C, H, W)
single_sv_ch = single_sv.reshape(shap_values.shape[1], -1).mean(axis=1)  # mean per channel

base_value = float(get_values(sample_obs[:50]).mean())  # approximate E[V(s)]

waterfall_exp = shap.Explanation(
    values=single_sv_ch,
    base_values=base_value,
    data=sample_obs[best_idx].reshape(shap_values.shape[1], -1).mean(axis=1),
    feature_names=CHANNEL_NAMES[:len(single_sv_ch)],
)
shap.plots.waterfall(waterfall_exp, show=False)
plt.title(f'SHAP waterfall — best observed state (V={all_values[best_idx]:.2f})')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'interpretability_shap_waterfall.png', dpi=120, bbox_inches='tight')
plt.show()
print('Waterfall: starting from E[V(s)]={base_value:.2f}, each channel contribution is shown.')
print('Red bars = positive contribution to V(s), blue bars = negative contribution.')


---
## 9 — Learning Curves from MLflow

Training metrics are pulled live from the MLflow `rl_training` experiment and plotted.
This demonstrates the **MLOps experiment tracking** workflow required by the project guidelines.


In [ ]:
# 8. Pull training metrics from MLflow and plot
import mlflow
import matplotlib.ticker as mticker

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = mlflow.tracking.MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

exp = client.get_experiment_by_name('rl_training')
if exp is None:
    print('No rl_training experiment found — run section 6 first.')
else:
    runs = client.search_runs(
        experiment_ids=[exp.experiment_id],
        order_by=['start_time DESC'],
        max_results=5,
    )
    print(f'Found {len(runs)} run(s). Plotting most recent…')
    run = runs[0]

    def load_metric(run, key):
        history = client.get_metric_history(run.info.run_id, key)
        if not history:
            return None, None
        steps  = [m.step for m in history]
        values = [m.value for m in history]
        return steps, values

    metrics = {
        'Mean reward':   'rollout/ep_rew_mean',
        'Mean length':   'rollout/ep_len_mean',
        'Policy loss':   'train/policy_loss',
        'Value loss':    'train/value_loss',
    }

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    for ax, (title, key) in zip(axes.flat, metrics.items()):
        steps, vals = load_metric(run, key)
        if steps:
            ax.plot(steps, vals, lw=1.2)
            ax.set_title(title)
            ax.set_xlabel('Timestep')
            ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
            ax.grid(True, alpha=0.3)
        else:
            ax.text(0.5, 0.5, f'No data for\n{key}', ha='center', va='center', transform=ax.transAxes)

    run_name = run.data.tags.get('mlflow.runName', run.info.run_id[:8])
    plt.suptitle(f'Learning curves — {run_name}', y=1.01, fontsize=13)
    plt.tight_layout()
    plt.savefig(REPORTS_DIR / 'learning_curve_reward.png', dpi=120, bbox_inches='tight')
    plt.show()

---
## 10 — Agent Benchmark: Direct-State vs Vision Pipeline

Final quantitative evaluation: compare the **direct-state agent** (reads game state perfectly)
against the **vision agent** (must infer state from raw pixel frames via the U-Net).
Expected gap reflects the information loss introduced by the segmentation step.


In [ ]:
# 9. Run benchmark — mirrors scripts/benchmark_agents.py
import subprocess, sys
result = subprocess.run(
    [sys.executable, str(ROOT / 'scripts' / 'benchmark_agents.py')],
    capture_output=True, text=True, cwd=str(ROOT),
)
print(result.stdout[-2000:] if result.stdout else '')
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

chart = REPORTS_DIR / 'benchmark_scores.png'
if chart.exists():
    display(IPImage(str(chart)))

---
## 11 — Conclusions

### Key Findings

| Finding | Detail |
|---------|--------|
| **RL agent learns** | MaskablePPO with action masking achieves a stable policy. Curriculum training (short → long episodes) accelerates convergence. |
| **Segmentation accuracy** | TinyU-Net reaches mIoU ≈ 0.64 on 13 classes. Walls and pellets are the dominant pixel classes. |
| **Slot-mask beats NN for pellets** | Deterministic coordinate sampling is faster and more reliable than the neural network for static objects. |
| **Vision gap** | Direct agent significantly outperforms vision agent, confirming information loss from imperfect segmentation. |
| **SHAP insights** | Walls and pellet channels dominate V(s) attribution. Ghost channels matter most when in proximity to Pac-Man. |


### Future Work
- Fine-tune the segmentation model on real Pac-Man screenshots.
- Implement procedurally generated maze levels for curriculum diversity.
- Apply LIME for local policy explanations at the action level.
- Explore model distillation: use the direct agent to supervise the vision agent (teacher-student).

---
*All runs are reproducible: `RANDOM_SEED = 42`, all hyperparameters and artifacts logged to MLflow.*
